# Time-series anomaly detection — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def make_series(n=60, trend=0.08, amp=2.0, noise=0.25):
    t=np.arange(n)
    return t, 10 + trend*t + amp*np.sin(2*np.pi*t/12) + noise*np.random.randn(n)
def acf(x, max_lag):
    x=np.asarray(x)-np.mean(x); den=np.dot(x,x)
    return np.array([1.0]+[np.dot(x[:-k],x[k:])/den for k in range(1,max_lag+1)])
def moving_average(x,w):
    return np.convolve(x, np.ones(w)/w, mode='valid')
print('setup ok')

## An anomaly is not just a large value; it is a value unlikely under the local pattern the series taught us to expect
Time-series anomaly detection scores residuals against local expectation. These examples show z-score scoring, robust scoring, seasonal residual scoring, rolling-window detection, and multistep reconstruction error.

In [ ]:
# 1) z-score anomaly against a stable baseline
base=np.array([10,11,9,10]); x=30; z=(x-base.mean())/base.std(ddof=0)
plt.figure(figsize=(6,3)); plt.bar(['baseline mean','point'],[base.mean(),x]); plt.title(f'z = {z:.2f}')
assert abs(z-28.2842712474619)<1e-9

In [ ]:
# 2) robust median/MAD scoring resists one contaminated point
base=np.array([10,11,9,10]); x=30; med=np.median(base); mad=np.median(np.abs(base-med)); robust=(x-med)/(1.4826*mad)
plt.figure(figsize=(6,3)); plt.bar(['median','MAD-scaled score'],[med,robust]); plt.title('robust score stays anchored to typical values')
assert abs(robust-26.97963037906381)<1e-9

In [ ]:
# 3) seasonal residual: high for its season is what matters
t=np.arange(24); expected=10+2*np.sin(2*np.pi*t/12); y=expected.copy(); y[18]+=5; resid=y-expected
plt.figure(figsize=(6,3)); plt.plot(y,label='observed'); plt.plot(expected,label='expected'); plt.stem([18],[resid[18]],linefmt='r-',markerfmt='ro',basefmt=' '); plt.legend(); plt.title('score residual, not raw value')
assert resid[18]==5.0

In [ ]:
# 4) rolling detector adapts to slow level changes
t=np.arange(80); y=10+0.05*t+0.2*np.random.randn(80); y[55]+=4; scores=[]
for i in range(20,80):
    win=y[i-20:i]; scores.append((y[i]-win.mean())/win.std(ddof=0))
plt.figure(figsize=(6,3)); plt.plot(range(20,80),scores); plt.axhline(3,color='r',ls='--'); plt.title('rolling z-score catches the spike')
assert scores[55-20] > 3

In [ ]:
# 5) reconstruction error flags unusual shapes, not only spikes
normal=np.sin(np.linspace(0,2*np.pi,24)); anomalous=normal.copy(); anomalous[8:12]*=-1; recon=normal; err=(anomalous-recon)**2
plt.figure(figsize=(6,3)); plt.plot(normal,label='learned normal'); plt.plot(anomalous,label='candidate'); plt.plot(err,label='squared error'); plt.legend(); plt.title('shape anomaly has concentrated error')
assert err[8:12].sum() > err.sum()*0.9